In [2]:
import os
import re
import sqlite3
import pandas as pd

In [30]:
import plotly.io as pio

pio.renderers.default = 'vscode' #'svg' # 'png'  #"notebook_connected" # "notebook" #  "jupyterlab"  # or "vscode"

In [4]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dash import Dash, dcc, html, Input, Output, callback
import nbformat

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
if os.environ['OPENAI_API_KEY']:
    print("API Key is set")

API Key is set


In [8]:
# Setting temperature=0 ensures deterministic, accurate SQL generation.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [9]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful analyst that answers in exactly two sentences."),
    ("human", "{question}")
])

# Chain them together and invoke
chain = prompt | llm | StrOutputParser()

# Execute the chain
response = chain.invoke({"question": "What are the core components of LangChain?"})
print(response)

The core components of LangChain include LLMs (Large Language Models), document loaders, and chains that facilitate the integration of various components for building applications. Additionally, it features memory management and agents that enhance interaction and decision-making capabilities.


#### Inspecting the tables in our database

In [10]:
DATABASE_PATH = 'nyc_taxi_analytics.db'

In [11]:
# Establish connection to SQLite file
con = sqlite3.connect(DATABASE_PATH)
cursor = con.cursor()

In [12]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = cursor.fetchall()
print("Tables in database:", [t[0] for t in tables])

Tables in database: ['daily_trips', 'reporting_data_trips']


#### Inspecting the schema of tables in our database

In [13]:
def get_schema_info(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    schema_info = []
    # Get all tables
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()
    for (table_name,) in tables:
        # Get columns for this table
        cursor.execute(f"PRAGMA table_info({table_name})")
        columns = cursor.fetchall()
        table_info = f"Table: {table_name}\n"
        table_info += "\n".join(f"  - {col[1]} ({col[2]})" for col in columns)
        schema_info.append(table_info)
    conn.close()
    return "\n\n".join(schema_info)

In [14]:
# Get the schema info
schema = get_schema_info(DATABASE_PATH)
print(schema)

Table: daily_trips
  - pickup_date (TEXT)
  - total_rides (INTEGER)
  - total_tip_amount (INTEGER)
  - total_fare_amount (INTEGER)
  - total_amount (INTEGER)

Table: reporting_data_trips
  - pickup_date (TIMESTAMP)
  - pickup_hour (INTEGER)
  - distance_bucket (TEXT)
  - payment_type (INTEGER)
  - Borough_PU (TEXT)
  - total_rides (INTEGER)
  - total_tip_amount (INTEGER)
  - total_fare_amount (INTEGER)
  - total_amount (INTEGER)


#### Executing SQL Queries on SQLite tables - when user writes SQL Queries manually.

In [15]:
def query_a_table(db_path,user_query):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute(user_query)

    output_data = cursor.fetchall()

    # print(f"Sample output: {cursor.fetchall()}")

     # Extract column names from the cursor description
    columns = [column[0] for column in cursor.description]

    # Convert to DataFrame
    output_df = pd.DataFrame(output_data, columns=columns)

    conn.close()
    return output_df

In [16]:
user_query = "SELECT * FROM daily_trips LIMIT 5;" # need to provide which table you want to query.
query_output = query_a_table(DATABASE_PATH,user_query)
query_output

,pickup_date,total_rides,total_tip_amount,total_fare_amount,total_amount
0,2026-02-01,118982,281275,2591585,3552174
1,2026-02-02,104913,316360,2401451,3360294
2,2026-02-03,116258,345777,2655762,3706912
3,2026-02-04,121766,363210,2731673,3832118
4,2026-02-05,126307,382169,2856422,4014331


In [17]:
# Get the schema info
user_query = "SELECT distance_bucket , SUM(total_rides) FROM reporting_data_trips GROUP BY distance_bucket;"
query_a_table(DATABASE_PATH,user_query)


,distance_bucket,SUM(total_rides)
0,Long (>5m),1952111
1,Medium (2-5m),2923862
2,Short (<2m),5165690


In [18]:
# Query to calculate a running total of rides over time.
cursor.execute("SELECT pickup_date, total_rides, SUM(total_rides) OVER(ORDER BY pickup_date) as running_sum FROM daily_trips ;")
# print(f"Sample output: {cursor.fetchall()}")

# Fetch data and load it into a DataFrame
output_data = cursor.fetchall()

# Print Header
print(f"{'pickup_date':<12} | {'total_rides':<11} | {'running_sum':<11}")
print("-" * 42)

# Print Rows
for row in output_data:
    print(f"{row[0]:<12} | {row[1]:<11} | {row[2]:<11}")

pickup_date  | total_rides | running_sum
------------------------------------------
2026-02-01   | 118982      | 118982     
2026-02-02   | 104913      | 223895     
2026-02-03   | 116258      | 340153     
2026-02-04   | 121766      | 461919     
2026-02-05   | 126307      | 588226     
2026-02-06   | 124790      | 713016     
2026-02-07   | 149601      | 862617     
2026-02-08   | 125961      | 988578     
2026-02-09   | 104462      | 1093040    
2026-02-10   | 111087      | 1204127    
2026-02-11   | 120858      | 1324985    
2026-02-12   | 133015      | 1458000    
2026-02-13   | 129723      | 1587723    
2026-02-14   | 129273      | 1716996    
2026-02-15   | 110333      | 1827329    
2026-02-16   | 80337       | 1907666    
2026-02-17   | 99811       | 2007477    
2026-02-18   | 112628      | 2120105    
2026-02-19   | 117388      | 2237493    
2026-02-20   | 119254      | 2356747    
2026-02-21   | 132329      | 2489076    
2026-02-22   | 82838       | 2571914    
2026-02-23   |

## Creating a Text to SQL Prompt

Now that we have our database set up, let's create a basic prompt for Text to SQL conversion. 
A good prompt should include:

- Clear instructions for what we want the model to do.
- The user's query.
- The database's schema, so the llm knows how to translate the user's query.

### INPUT GUARDRAILS

Intent Validation & Disambiguation (Pre-Parsing)

Before sending the question to the SQL generator, pass it through an **intent-classification layer.**

 **Out-of-Bounds Filter**: If a user asks "What is the weather like?" but your database holds e-commerce data, the model should immediately return a friendly error or a clarifying question instead of attempting to generate a SQL query.

In [19]:
def parse_question(user_question, schema):
        """Parse user question and identify relevant tables and columns."""

        prompt = ChatPromptTemplate.from_messages([
            ("system", '''You are a data analyst that can help summarize SQL tables and parse user questions about a database. 
                        Given the question and database schema, identify the relevant tables and columns. 
                        If the question is not relevant to the database or if there is not enough information to answer the question, set is_relevant to false.

                Your response should be in the following JSON format:
                {{
                    "is_relevant": boolean,
                    "relevant_tables": [
                        {{
                            "table_name": string,
                            "columns": [string],
                            "relevant_columns": [string]
                        }}
                    ]
                }}

                 "relevant_columns": It should contain only the columns that are relevant to the question and contain nouns or names, for example, the column "Artist name" contains nouns relevant to the question "What are the top selling artists?", but the column "Artist ID" is not relevant because it does not contain a noun. Do not include columns that contain numbers.
                '''),
                            ("human", "===Database schema:\n{schema}\n\n===User question:\n{user_question}\n\nIdentify relevant tables and columns:")
                        ])

        output_parser = JsonOutputParser()
        
        # print(output_parser)

        chain = prompt | llm | output_parser

        # response = llm.invoke(prompt, schema=schema, user_question=user_question)

        # Pass the variables directly into the chain invoke method
        parsed_response = chain.invoke({"schema": schema, "user_question": user_question})

        return {"parsed_question": parsed_response}

#### Testing parse_question

In [35]:
parse_question("Show me the total rides volume for each borough", schema)

{'parsed_question': {'is_relevant': False, 'relevant_tables': []}}

In [46]:
parse_question("Show me the total rides volume for each borough", schema)

{'parsed_question': {'is_relevant': True,
  'relevant_tables': [{'table_name': 'reporting_data_trips',
    'columns': ['pickup_date',
     'pickup_hour',
     'distance_bucket',
     'payment_type',
     'Borough_PU',
     'total_rides',
     'total_tip_amount',
     'total_fare_amount',
     'total_amount'],
    'relevant_columns': ['Borough_PU', 'total_rides']}]}}

In [44]:
parse_question("Show me the total rides volume for each US state", schema)

{'parsed_question': {'is_relevant': False, 'relevant_tables': []}}

In [51]:
parse_question("Show me the total rides volume for queens", schema)

{'parsed_question': {'is_relevant': True,
  'relevant_tables': [{'table_name': 'reporting_data_trips',
    'columns': ['pickup_date',
     'pickup_hour',
     'distance_bucket',
     'payment_type',
     'Borough_PU',
     'total_rides',
     'total_tip_amount',
     'total_fare_amount',
     'total_amount'],
    'relevant_columns': ['Borough_PU', 'total_rides']}]}}

In [52]:
parse_question("Show me the total rides volume for new jersey", schema)

{'parsed_question': {'is_relevant': True,
  'relevant_tables': [{'table_name': 'reporting_data_trips',
    'columns': ['pickup_date',
     'pickup_hour',
     'distance_bucket',
     'payment_type',
     'Borough_PU',
     'total_rides',
     'total_tip_amount',
     'total_fare_amount',
     'total_amount'],
    'relevant_columns': ['Borough_PU', 'total_rides']}]}}

In [53]:
parse_question("Show me the total rides volume for ohio", schema)

{'parsed_question': {'is_relevant': False, 'relevant_tables': []}}

In [29]:
parse_question("Show me the total rides volume for each day", schema)

{'parsed_question': {'is_relevant': True,
  'relevant_tables': [{'table_name': 'daily_trips',
    'columns': ['pickup_date', 'total_rides'],
    'noun_columns': ['pickup_date']}]}}

In [30]:
parse_question("what is the tip percentage for each day", schema)

{'parsed_question': {'is_relevant': True,
  'relevant_tables': [{'table_name': 'daily_trips',
    'columns': ['pickup_date',
     'total_rides',
     'total_tip_amount',
     'total_fare_amount',
     'total_amount'],
    'noun_columns': ['pickup_date', 'total_tip_amount']}]}}

In [36]:
parse_question("What is the weather like?", schema)

{'parsed_question': {'is_relevant': False, 'relevant_tables': []}}

In [37]:
parse_question("Which day has the best day?",schema)

{'parsed_question': {'is_relevant': True,
  'relevant_tables': [{'table_name': 'daily_trips',
    'columns': ['pickup_date',
     'total_rides',
     'total_tip_amount',
     'total_fare_amount',
     'total_amount'],
    'noun_columns': ['pickup_date']}]}}

In [38]:
parse_question("Find total rides volume in the last 7 days bases on the maxium date as per the dataset", schema)

{'parsed_question': {'is_relevant': True,
  'relevant_tables': [{'table_name': 'daily_trips',
    'columns': ['pickup_date', 'total_rides'],
    'noun_columns': ['pickup_date']}]}}

In [39]:
parse_question("What is the total rides volume for Sundays?", schema)

{'parsed_question': {'is_relevant': True,
  'relevant_tables': [{'table_name': 'daily_trips',
    'columns': ['pickup_date',
     'total_rides',
     'total_tip_amount',
     'total_fare_amount',
     'total_amount'],
    'noun_columns': ['pickup_date', 'total_rides']}]}}

### Inject Schema Awareness (The Prompt)

An LLM cannot guess your database structure. You must explicitly provide its schema as context so it only writes queries relevant to your tables and columns.

• Table Schemas: Provide CREATE TABLE definitions, column names, and data types.

• Relationships: Explicitly map primary and foreign keys so the model knows how to JOIN tables.

• Sample Data: Provide 1-2 sample rows from each table to give the model context on how the data is formatted for complex database schemas.

In [20]:
# Generate Prompt for SQL generation
prompt_template = prompt_template = ChatPromptTemplate.from_messages([
        ("system", "You are an AI assistant that converts natural language queries into SQL."),
        ("human", """Given the following SQLite3 database schema:
    <schema>
    {schema}
    </schema>

    Convert the following natural language query into SQL:
    QUERY
    {question}
    QUERYEND
         
    Provide only the SQL query for SQLite3 database in your response, enclosed within <sql> tags.""")
    ])

chain = prompt_template | llm | StrOutputParser()

def generate_sql(db_schema, user_question):

    result = chain.invoke({
        "schema": db_schema, 
        "question": user_question
    })

    if "<sql>" in result and "</sql>" in result:
        return result.split("<sql>")[1].split("</sql>")[0].strip()
    return result.strip()


In [21]:
def execute_sql_query(sql_query):
    # Connect to your SQLite database file.
    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()
    
    # Execute the query
    cursor.execute(sql_query)

    # Fetch all rows returned by the query
    results = cursor.fetchall()

    # Process and print the results
    ##print("Query Results:")

    ##for row in results:
    ##    print(row)
    # print(f"Sample output: {cursor.fetchall()}")

    # Extract column names from the cursor description
    columns = [column[0] for column in cursor.description]

    # Convert to DataFrame
    df = pd.DataFrame(results, columns=columns)

    cursor.close()
    conn.close()
    
    return df

In [22]:
def generate_plotly_answer(question,df): # Query result -> pandas dataframe

    # Define the LLM Prompt Template

    PLOTLY_GENERATOR_PROMPT = """
    You are an expert Python data visualization assistant specializing in Plotly.
    Your task is to write ONLY executable Python code that creates an interactive Plotly figure based on the user's question, the executed SQL query, and the structure of the resulting DataFrame.

    CRITICAL INSTRUCTIONS:
    - Assume a pandas DataFrame named `df` ALREADY exists in the local namespace.
    - DO NOT recreate `df` or hardcode sample data.
    - The columns available in `df` are exactly: {columns_list}
    - The data types of the columns are:
    {dtypes_info}

    CODE REQUIREMENTS:
    - You must create the final figure and assign it to a variable named `fig` (e.g., `fig = px.bar(...)` or `fig = go.Figure(...)`).
    - Do not wrap the code in a function.
    - Do not call `fig.show()`.
    - Return ONLY valid, raw Python code inside a markdown block. Do not include introductory text, explanations, or commentary.
    - If the dataframe contains a single metric/number, display that in form of a metric.

    Available chart types and their use cases:
    - Bar Graphs: Best for comparing categorical data or showing changes over time when categories are discrete and the number of categories is more than 2. Use for questions like "What are the sales figures for each product?" or "How does the population of cities compare? or "What percentage of each city is male?"
    - Horizontal Bar Graphs: Best for comparing categorical data or showing changes over time when the number of categories is small or the disparity between categories is large. Use for questions like "Show the revenue of A and B?" or "How does the population of 2 cities compare?" or "How many men and women got promoted?" or "What percentage of men and what percentage of women got promoted?" when the disparity between categories is large.
    - Scatter Plots: Useful for identifying relationships or correlations between two numerical variables or plotting distributions of data. Best used when both x axis and y axis are continuous. Use for questions like "Plot a distribution of the fares (where the x axis is the fare and the y axis is the count of people who paid that fare)" or "Is there a relationship between advertising spend and sales?" or "How do height and weight correlate in the dataset? Do not use it for questions that do not have a continuous x axis."
    - Pie Charts: Ideal for showing proportions or percentages within a whole. Use for questions like "What is the market share distribution among different companies?" or "What percentage of the total revenue comes from each product?"
    - Line Graphs: Best for showing trends and distributionsover time. Best used when both x axis and y axis are continuous. Used for questions like "How have website visits changed over the year?" or "What is the trend in temperature over the past decade?". Do not use it for questions that do not have a continuous x axis or a time based x axis.

    Consider these types of questions when recommending a visualization:
    1. Aggregations and Summarizations (e.g., "What is the average revenue by month?" - Line Graph)
    2. Comparisons (e.g., "Compare the sales figures of Product A and Product B over the last year." - Line or Column Graph)
    3. Plotting Distributions (e.g., "Plot a distribution of the age of users" - Scatter Plot)
    4. Trends Over Time (e.g., "What is the trend in the number of active users over the past year?" - Line Graph)
    5. Proportions (e.g., "What is the market share of the products?" - Pie Chart)
    6. Correlations (e.g., "Is there a correlation between marketing spend and revenue?" - Scatter Plot)

    CONTEXT:
    User Question: {question}
    DataFrame Head Preview:
    {df_preview}
    """
    # Your response should be in the following JSON format:
    #             {{
    #                 "sql_query": string,
    #                 "chart_type": string
    #             }}

    # Extract structural metadata to safely feed to the prompt

    columns_list = list(df.columns)
    dtypes_info = df.dtypes.to_string()
    df_preview = df.head(3).to_string()

    prompt_template = ChatPromptTemplate.from_template(PLOTLY_GENERATOR_PROMPT)

    chain =  prompt_template | llm 

    # Pass the variables directly into the chain invoke method
    response = chain.invoke({"columns_list": columns_list, "dtypes_info":dtypes_info, "question": question, "df_preview":df_preview})

    # prompt = prompt_template.format(
    #     question=question,
    #     # sql_query=sql_query,
    #     columns_list=columns_list,
    #     dtypes_info=dtypes_info,
    #     df_preview=df_preview
    # )

     # Invoke the model
    # response = llm.invoke(prompt)
    raw_content = response.content if hasattr(response, 'content') else str(response)

    # Regex extract to parse code out of markdown blocks reliably
    code_match = re.search(r"```python\s*(.*?)\s*```", raw_content, re.DOTALL)
    if code_match:
        return code_match.group(1).strip()
    
    return raw_content.strip()

In [23]:
def display_visual(code_str, result_df):
    # Generate the code string
    # code_str = generate_plotly_answer(question, result_df)

    # Setup isolated context dictionary with your target dataframe
    execution_context = {"df":result_df}

    # Execute the generated code
    exec(code_str, {}, execution_context)

    # Extract your fig object
    fig = execution_context.get("fig")
    fig.show()

In [24]:
def query_to_sql_to_chart(user_question, db_path):

    schema = get_schema_info(db_path)
    captured_response = parse_question(user_question, schema)
    print(captured_response)

    if not captured_response['parsed_question']['is_relevant']:
        print('Sorry, I do not have relevant information to answer your question.')
    else:
        # Generate SQL
        output_sql = generate_sql(schema, user_question)
    
        print("Generated SQL:")
        print(output_sql)

        # Execute SQL on the db table
        result_df = execute_sql_query(output_sql)

        print("Displaying few records from the result generated by SQL Query")
        # Execute SQL on the db table
        print(result_df.head(10))

        # Generate plotly code for a visual
        output_plotly = generate_plotly_answer(user_question,result_df)

        print('Generated_plotly_code:')
        print(output_plotly)

        # Display the chart/visual
        display_visual(output_plotly, result_df)
        return result_df

In [31]:
question = "Can you show me trend line for total rides for manhattan and brooklyn"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
SELECT pickup_date, SUM(total_rides) AS total_rides
FROM reporting_data_trips
WHERE Borough_PU IN ('Manhattan', 'Brooklyn')
GROUP BY pickup_date
ORDER BY pickup_date;
Displaying few records from the result generated by SQL Query
           pickup_date  total_rides
0  2026-01-01 00:00:00        92513
1  2026-01-02 00:00:00        80775
2  2026-01-03 00:00:00        92402
3  2026-01-04 00:00:00        76356
4  2026-01-05 00:00:00        77345
5  2026-01-06 00:00:00        88819
6  2026-01-07 00:00:00        95203
7  2026-01-08 00:00:00       101210
8  2026-01-09 00:00:00       104037
9  2026-01-10 00:00:00       129629
Generated_plotly_code:
imp

In [36]:
question = "Can you show me percentange difference in rides volume for each borough month over month along with the actual rides volume"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
WITH monthly_rides AS (
    SELECT 
        strftime('%Y-%m', pickup_date) AS month,
        Borough_PU,
        SUM(total_rides) AS total_rides
    FROM 
        reporting_data_trips
    GROUP BY 
        month, Borough_PU
),
rides_difference AS (
    SELECT 
        current.month,
        current.Borough_PU,
        current.total_rides AS current_rides,
        COALESCE(previous.total_rides, 0) AS previous_rides,
        (current.total_rides - COALESCE(previous.total_rides, 0)) * 100.0 / NULLIF(previous.total_rides, 0) AS percentage_difference
    FROM 
        monthly_rides AS current
    LEFT JOIN 
        monthly_rides AS previous 
    ON

In [37]:
output_df

,month,Borough_PU,current_rides,previous_rides,percentage_difference
0,2026-01,None,499,0,None
1,2026-01,Bronx,33638,0,None
2,2026-01,Brooklyn,136610,0,None
3,2026-01,EWR,24,0,None
4,2026-01,Manhattan,2871761,0,None
5,2026-01,Queens,308017,0,None
6,2026-01,Staten Island,424,0,None
7,2026-01,Unknown,3812,0,None
8,2026-02,None,421,0,None
9,2026-02,Bronx,31292,0,None


In [38]:
question = "Can you show me percentange difference in rides volume for manhattan month over month along with the actual rides volume"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
WITH monthly_rides AS (
    SELECT 
        strftime('%Y-%m', pickup_date) AS month,
        SUM(total_rides) AS total_rides
    FROM 
        reporting_data_trips
    WHERE 
        Borough_PU = 'Manhattan'
    GROUP BY 
        month
),
rides_with_difference AS (
    SELECT 
        month,
        total_rides,
        LAG(total_rides) OVER (ORDER BY month) AS previous_month_rides
    FROM 
        monthly_rides
)
SELECT 
    month,
    total_rides,
    CASE 
        WHEN previous_month_rides IS NULL THEN NULL
        ELSE ((total_rides - previous_month_rides) * 100.0 / previous_month_rides)
    END AS percentage_difference
FROM 
    rides_wi

In [39]:
question = "Can you show me percentange difference in rides volume for manhattan and brooklyn month over month along with the actual rides volume"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
WITH monthly_rides AS (
    SELECT 
        strftime('%Y-%m', pickup_date) AS month,
        Borough_PU,
        SUM(total_rides) AS total_rides
    FROM reporting_data_trips
    WHERE Borough_PU IN ('Manhattan', 'Brooklyn')
    GROUP BY month, Borough_PU
),
rides_difference AS (
    SELECT 
        a.month,
        a.total_rides AS manhattan_rides,
        b.total_rides AS brooklyn_rides,
        (a.total_rides - b.total_rides) * 100.0 / NULLIF(b.total_rides, 0) AS percentage_difference
    FROM monthly_rides a
    JOIN monthly_rides b ON a.month = b.month AND a.Borough_PU = 'Manhattan' AND b.Borough_PU = 'Brooklyn'
)
SELECT 
    month,
    m

In [40]:
output_df

,month,manhattan_rides,brooklyn_rides,percentage_difference
0,2026-01,2871761,136610,2002.160164
1,2026-02,2618337,130431,1907.449916


In [79]:
question = "Can you show me percentange change in rides volume for manhattan and brooklyn monthly respectively"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'daily_trips', 'columns': ['pickup_date', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'total_rides']}, {'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
SELECT 
    strftime('%Y-%m', pickup_date) AS month,
    SUM(CASE WHEN Borough_PU = 'Manhattan' THEN total_rides ELSE 0 END) AS manhattan_rides,
    SUM(CASE WHEN Borough_PU = 'Brooklyn' THEN total_rides ELSE 0 END) AS brooklyn_rides,
    (SUM(CASE WHEN Borough_PU = 'Manhattan' THEN total_rides ELSE 0 END) - 
     LAG(SUM(CASE WHEN Borough_PU = 'Manhattan' THEN total_rides ELSE 0 END)) OVER (ORDER BY strftime('%Y-%m', pickup_date))) * 100.0 /
     NULLIF(LAG(SUM(C

In [80]:
output_df

,month,manhattan_rides,brooklyn_rides,manhattan_percentage_change,brooklyn_percentage_change
0,2026-01,2871761,136610,NaN,NaN
1,2026-02,2618337,130431,-8.824690,-4.523095
2,2026-03,3124810,145865,19.343308,11.833076


In [81]:
question = "Can you show me percentange change in rides volume for manhattan and brooklyn monthly respectively, the sql output should look something like this (month,borough,rides volume, percentage change)"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'daily_trips', 'columns': ['pickup_date', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'total_rides']}, {'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
WITH monthly_rides AS (
    SELECT 
        strftime('%Y-%m', pickup_date) AS month,
        Borough_PU,
        SUM(total_rides) AS rides_volume
    FROM 
        reporting_data_trips
    WHERE 
        Borough_PU IN ('Manhattan', 'Brooklyn')
    GROUP BY 
        month, Borough_PU
),
rides_change AS (
    SELECT 
        month,
        Borough_PU,
        rides_volume,
        LAG(rides_volume) OVER (PARTITION BY Borough_PU ORDER BY month) AS previous_rides_volu

In [82]:
output_df

,month,Borough_PU,rides_volume,percentage_change
0,2026-01,Brooklyn,136610,NaN
1,2026-01,Manhattan,2871761,NaN
2,2026-02,Brooklyn,130431,-4.523095
3,2026-02,Manhattan,2618337,-8.824690
4,2026-03,Brooklyn,145865,11.833076
5,2026-03,Manhattan,3124810,19.343308


In [53]:
import numpy as np

In [54]:
# Define the dataset as a dictionary
data = {
    "month": ["2026-01", "2026-01", "2026-02", "2026-02"],
    "Borough_PU": ["Brooklyn", "Manhattan", "Brooklyn", "Manhattan"],
    "rides_volume":[136610,2871761,130431,2618337],
    "percentage_change": [np.nan, np.nan, -4.523095, -8.824690],
}

# Create the DataFrame
df = pd.DataFrame(data)


In [63]:
df

,month,Borough_PU,rides_volume,percentage_change
0,2026-01,Brooklyn,136610,NaN
1,2026-01,Manhattan,2871761,NaN
2,2026-02,Brooklyn,130431,-4.523095
3,2026-02,Manhattan,2618337,-8.824690


In [64]:
df.dropna()

,month,Borough_PU,rides_volume,percentage_change
2,2026-02,Brooklyn,130431,-4.523095
3,2026-02,Manhattan,2618337,-8.824690


In [56]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Create a subplot figure with 1 row and 2 columns
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=("Total Rides Volume", "Month-over-Month % Change")
)

# 2. Add Volume Bar Chart (Left Side)
for borough in df['Borough_PU'].unique():
    df_borough = df[df['Borough_PU'] == borough]
    fig.add_trace(
        go.Bar(
            x=df_borough['month'], 
            y=df_borough['rides_volume'], 
            name=f"{borough} (Volume)"
        ),
        row=1, col=1
    )

# 3. Add Percentage Change Line Chart (Right Side)
# Dropping NaN for the line plot to connect points cleanly
df_clean = df.dropna(subset=['percentage_change'])
for borough in df_clean['Borough_PU'].unique():
    df_borough = df_clean[df_clean['Borough_PU'] == borough]
    fig.add_trace(
        go.Scatter(
            x=df_borough['month'], 
            y=df_borough['percentage_change'], 
            mode='lines+markers',
            name=f"{borough} (% Change)"
        ),
        row=1, col=2
    )

# 4. Final Layout Adjustments
fig.update_layout(
    title_text="Borough Ride Metrics Comparison", 
    barmode='group',
    hovermode='x unified'
)
fig.show()


In [57]:
fig = px.bar(df, x='month', y='rides_volume', color='Borough_PU', barmode='group')
fig.show()

In [83]:
question = "Can you show me percentange change in rides volume for each borough monthly respectively, the sql output should look something like this (month,borough,rides volume, percentage change)"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
WITH monthly_rides AS (
    SELECT 
        strftime('%Y-%m', pickup_date) AS month,
        Borough_PU,
        SUM(total_rides) AS rides_volume
    FROM 
        reporting_data_trips
    GROUP BY 
        month, Borough_PU
),
rides_change AS (
    SELECT 
        month,
        Borough_PU,
        rides_volume,
        LAG(rides_volume) OVER (PARTITION BY Borough_PU ORDER BY month) AS previous_rides_volume
    FROM 
        monthly_rides
)
SELECT 
    month,
    Borough_PU,
    rides_volume,
    CASE 
        WHEN previous_rides_volume IS NULL THEN NULL
        ELSE (rides_volume - previous_rides_volume) * 100.0 / previous_rides_volume 
    

In [84]:
output_df

,month,Borough_PU,rides_volume,percentage_change
0,2026-01,None,499,NaN
1,2026-01,Bronx,33638,NaN
2,2026-01,Brooklyn,136610,NaN
3,2026-01,EWR,24,NaN
4,2026-01,Manhattan,2871761,NaN
5,2026-01,Queens,308017,NaN
6,2026-01,Staten Island,424,NaN
7,2026-01,Unknown,3812,NaN
8,2026-02,None,421,-15.631263
9,2026-02,Bronx,31292,-6.974255


In [85]:
question = "Can you show me percentange change in rides volume for each borough monthly respectively, the sql output should look something like this (month,borough,rides volume, percentage change, the visual should plot percentage changes for each borough and provide rides volume as label for each month)"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
WITH monthly_rides AS (
    SELECT 
        strftime('%Y-%m', pickup_date) AS month,
        Borough_PU,
        SUM(total_rides) AS rides_volume
    FROM 
        reporting_data_trips
    GROUP BY 
        month, Borough_PU
),
rides_change AS (
    SELECT 
        month,
        Borough_PU,
        rides_volume,
        LAG(rides_volume) OVER (PARTITION BY Borough_PU ORDER BY month) AS previous_rides_volume
    FROM 
        monthly_rides
)
SELECT 
    month,
    Borough_PU,
    rides_volume,
    CASE 
        WHEN previous_rides_volume IS NULL THEN NULL
        ELSE (rides_volume - previous_rides_volume) * 100.0 / previous_rides_volume 
    

In [86]:
output_df 

,month,Borough_PU,rides_volume,percentage_change
0,2026-01,None,499,NaN
1,2026-01,Bronx,33638,NaN
2,2026-01,Brooklyn,136610,NaN
3,2026-01,EWR,24,NaN
4,2026-01,Manhattan,2871761,NaN
5,2026-01,Queens,308017,NaN
6,2026-01,Staten Island,424,NaN
7,2026-01,Unknown,3812,NaN
8,2026-02,None,421,-15.631263
9,2026-02,Bronx,31292,-6.974255


In [87]:
question = "Can you show me percentange share of rides for each borough monthly, the sql output should look something like this (month,borough,rides volume, percentage share), the visual should plot percentage share of rides in each borough for each month)"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
SELECT 
    strftime('%Y-%m', pickup_date) AS month,
    Borough_PU AS borough,
    SUM(total_rides) AS rides_volume,
    (SUM(total_rides) * 100.0 / SUM(SUM(total_rides)) OVER (PARTITION BY strftime('%Y-%m', pickup_date))) AS percentage_share
FROM 
    reporting_data_trips
GROUP BY 
    month, borough
ORDER BY 
    month, borough;
Displaying few records from the result generated by SQL Query
     month        borough  rides_volume  percentage_share
0  2026-01           None           499          0.014874
1  2026-01          Bronx         33638          1.002687
2  2026-01       Brooklyn        136610          4.072094
3  2026-01            E

In [88]:
question = "Can you show me percentange share of rides for each borough monthly, the sql output should look something like this (month,borough,rides volume, percentage share), the visual should plot percentage share of rides in each borough for each month in a pie chart)"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
SELECT 
    strftime('%Y-%m', pickup_date) AS month,
    Borough_PU,
    SUM(total_rides) AS rides_volume,
    (SUM(total_rides) * 100.0 / SUM(SUM(total_rides)) OVER (PARTITION BY strftime('%Y-%m', pickup_date))) AS percentage_share
FROM 
    reporting_data_trips
GROUP BY 
    month, Borough_PU
ORDER BY 
    month, Borough_PU;
Displaying few records from the result generated by SQL Query
     month     Borough_PU  rides_volume  percentage_share
0  2026-01           None           499          0.014874
1  2026-01          Bronx         33638          1.002687
2  2026-01       Brooklyn        136610          4.072094
3  2026-01            EWR   

In [51]:
question = "Can you show me revenue for each payment type monthly, the sql output should look something like this (month,payment type,revenue)"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'payment_type', 'total_amount']}]}}
Generated SQL:
SELECT strftime('%Y-%m', pickup_date) AS month, 
       payment_type, 
       SUM(total_amount) AS revenue
FROM reporting_data_trips
GROUP BY month, payment_type
ORDER BY month, payment_type;
Displaying few records from the result generated by SQL Query
     month  payment_type   revenue
0  2026-01             0  28697607
1  2026-01             1  62949353
2  2026-01             2   7091402
3  2026-01             3    213550
4  2026-01             4    765843
5  2026-02             0  28773876
6  2026-02             1  57510543
7  2026-02             2   6212908
8  2026-02             3    156377
9  2026-02             4    515989


In [89]:
question = "Show me tip percentage for each borough for each month."
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'Borough_PU', 'total_rides', 'total_tip_amount']}]}}
Generated SQL:
SELECT 
    strftime('%Y-%m', pickup_date) AS month,
    Borough_PU,
    SUM(total_tip_amount) * 1.0 / SUM(total_fare_amount) * 100 AS tip_percentage
FROM 
    reporting_data_trips
GROUP BY 
    month, Borough_PU
ORDER BY 
    month, Borough_PU;
Displaying few records from the result generated by SQL Query
     month     Borough_PU  tip_percentage
0  2026-01           None        9.071268
1  2026-01          Bronx        0.371692
2  2026-01       Brooklyn        1.419525
3  2026-01            EWR       11.179111
4  2026-01      Manhattan       13.799703
5  2026-01         Queens       14.277793
6  2026-01  Staten Island    

In [90]:
output_df

,month,Borough_PU,tip_percentage
0,2026-01,None,9.071268
1,2026-01,Bronx,0.371692
2,2026-01,Brooklyn,1.419525
3,2026-01,EWR,11.179111
4,2026-01,Manhattan,13.799703
5,2026-01,Queens,14.277793
6,2026-01,Staten Island,0.973787
7,2026-01,Unknown,19.151445
8,2026-02,None,9.667310
9,2026-02,Bronx,0.389209


### KEY OBSERVATIONS

**1. LLMs almost always default to bar charts, line grapghs, ignoring pie charts even when the data is perfectly suited for one.**

This bias happens due to how LLMs are trained and how they reason. LLMs are heavily trained on programming documentations, data science tutorials, and visualization guidelines. Pie charts are widely disliked and actively discouraged.
If you want your agent to actually use pie charts when appropriate, you cannot rely on a generic prompt such as - pick the best chart. You have to explicityly engineer the capability.

- Implement a Visual Schema Validator.
- Explicit Few-shot prompting.

**2. The sql query generated is as good as the user question/instruction provided.**

The results are better when user explicitly mentions what the sql output should look like along with what the visual should capture when sql output has multiple measures/values that can be visualied.

**3. When a user question requires filtering for date columns,the user needs to mention the full date i.e Jan 20, 2026.**

It filters for a random date year when you only specify Jan 20th, even though the it is intuitive that the year is 2026.


##### Best Practices

• Pass Summaries, Not Full Tables: For large SQL outputs, pass df.head(10) or summary stats rather than thousands of rows to respect LLM token limits and reduce latency.

• Constrain Chart Types: Instruct the LLM on preferred chart types (e.g., "use bar charts for categorical comparisons, line charts for time-series").



In [55]:
question = "Are riders giving more tip or not?"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'daily_trips', 'columns': ['pickup_date', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'total_tip_amount']}, {'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'total_tip_amount']}]}}
Generated SQL:
SELECT 
    AVG(total_tip_amount) AS average_tip_amount_daily,
    AVG(total_tip_amount) AS average_tip_amount_reporting
FROM 
    daily_trips, reporting_data_trips;


OperationalError: ambiguous column name: total_tip_amount

In [56]:
question = "Identify the top 5 hours of the day when we saw maximum fares in Jan 2026"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'pickup_hour', 'total_fare_amount']}]}}
Generated SQL:
SELECT pickup_hour, SUM(total_fare_amount) AS total_fare
FROM reporting_data_trips
WHERE pickup_date >= '2026-01-01' AND pickup_date < '2026-02-01'
GROUP BY pickup_hour
ORDER BY total_fare DESC
LIMIT 5;
Displaying few records from the result generated by SQL Query
   pickup_hour  total_fare
0           15     4436782
1           21     4167337
2           20     4135100
3           14     4130035
4           17     4121191


In [57]:
question = "Identify the top 5 hours of the day when we saw maximum fares in Jan 2026, the visual should only show the hours sorted in descending order"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'pickup_hour', 'total_fare_amount']}]}}
Generated SQL:
SELECT pickup_hour, SUM(total_fare_amount) AS total_fare
FROM reporting_data_trips
WHERE pickup_date >= '2026-01-01' AND pickup_date < '2026-02-01'
GROUP BY pickup_hour
ORDER BY total_fare DESC
LIMIT 5;
Displaying few records from the result generated by SQL Query
   pickup_hour  total_fare
0           15     4436782
1           21     4167337
2           20     4135100
3           14     4130035
4           17     4121191


In [58]:
question = "Can you show me rides volume for each state in the US"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': False, 'relevant_tables': []}}
Sorry, I do not have relevant information to answer your question.


In [59]:
question = "Identify the top 5 hours of the day when we saw maximum fares in Jan 2026. How does that compare with Feb 2026. Draw Jan vs Feb bar chart showing fares, ride counts vs month in a single plot"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)


{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'pickup_hour', 'total_rides', 'total_fare_amount']}]}}
Generated SQL:
WITH January_Fares AS (
    SELECT pickup_hour, SUM(total_fare_amount) AS total_fare, SUM(total_rides) AS total_rides
    FROM reporting_data_trips
    WHERE pickup_date >= '2026-01-01' AND pickup_date < '2026-02-01'
    GROUP BY pickup_hour
    ORDER BY total_fare DESC
    LIMIT 5
),
February_Fares AS (
    SELECT pickup_hour, SUM(total_fare_amount) AS total_fare, SUM(total_rides) AS total_rides
    FROM reporting_data_trips
    WHERE pickup_date >= '2026-02-01' AND pickup_date < '2026-03-01'
    GROUP BY pickup_hour
    ORDER BY total_fare DESC
    LIMIT 5
)
SELECT 'January' AS month, pickup_hour, total_fare, total_ride

In [60]:
question = "Can you show rides volume trend and moving average for manhattan"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'daily_trips', 'columns': ['pickup_date', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'total_rides']}, {'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'total_rides', 'Borough_PU']}]}}
Generated SQL:
SELECT pickup_date, 
       SUM(total_rides) AS total_rides,
       AVG(SUM(total_rides)) OVER (ORDER BY pickup_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS moving_average
FROM reporting_data_trips
WHERE Borough_PU = 'Manhattan'
GROUP BY pickup_date
ORDER BY pickup_date;
Displaying few records from the result generated by SQL Query
           pickup_date  total_rides  moving_average
0  2026-01-01 00:00:00        85583    85583.000000
1  2026-01-02 00:00:00   

In [61]:
output_df

,pickup_date,total_rides,moving_average
0,2026-01-01 00:00:00,85583,85583.000000
1,2026-01-02 00:00:00,77268,81425.500000
2,2026-01-03 00:00:00,88401,83750.666667
3,2026-01-04 00:00:00,72055,80826.750000
4,2026-01-05 00:00:00,73959,79453.200000
5,2026-01-06 00:00:00,85273,80423.166667
6,2026-01-07 00:00:00,91444,81997.571429
7,2026-01-08 00:00:00,97409,83687.000000
8,2026-01-09 00:00:00,99833,86910.571429
9,2026-01-10 00:00:00,123759,91961.714286


In [91]:
question = "Plot the distribution of rides over time."
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'daily_trips', 'columns': ['pickup_date', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'total_rides']}, {'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'total_rides']}]}}
Generated SQL:
SELECT pickup_date, SUM(total_rides) AS total_rides
FROM daily_trips
GROUP BY pickup_date
ORDER BY pickup_date;
Displaying few records from the result generated by SQL Query
  pickup_date  total_rides
0  2026-02-01       118982
1  2026-02-02       104913
2  2026-02-03       116258
3  2026-02-04       121766
4  2026-02-05       126307
5  2026-02-06       124790
6  2026-02-07       149601
7  2026-02-08       125961
8  2026-02-09       104462
9  2026-02-10       111087
Generated_p

In [92]:
question = "Plot the distribution of rides."
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'daily_trips', 'columns': ['pickup_date', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'total_rides']}, {'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'total_rides']}]}}
Generated SQL:
SELECT total_rides, COUNT(*) AS ride_count
FROM daily_trips
GROUP BY total_rides
ORDER BY total_rides;
Displaying few records from the result generated by SQL Query
   total_rides  ride_count
0            2           1
1        21874           1
2        80337           1
3        82838           1
4        94625           1
5        99811           1
6       104462           1
7       104913           1
8       110333           1
9       111087           1
Generated_plotly_cod

In [93]:
question = "Plot the distribution of rides in Manhattan and Brooklyn."
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['Borough_PU', 'total_rides']}]}}
Generated SQL:
SELECT Borough_PU, SUM(total_rides) AS total_rides
FROM reporting_data_trips
WHERE Borough_PU IN ('Manhattan', 'Brooklyn')
GROUP BY Borough_PU;
Displaying few records from the result generated by SQL Query
  Borough_PU  total_rides
0   Brooklyn       412906
1  Manhattan      8614908
Generated_plotly_code:
import plotly.express as px

fig = px.bar(df, x='Borough_PU', y='total_rides', title='Distribution of Rides in Manhattan and Brooklyn')


In [94]:
question = "Is there a pattern for tip percentage for hours of the day."
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_hour', 'total_tip_amount', 'total_fare_amount']}]}}
Generated SQL:
SELECT pickup_hour, 
       SUM(total_tip_amount) * 1.0 / SUM(total_fare_amount) AS tip_percentage
FROM reporting_data_trips
GROUP BY pickup_hour
ORDER BY pickup_hour;
Displaying few records from the result generated by SQL Query
   pickup_hour  tip_percentage
0            0        0.101270
1            1        0.091086
2            2        0.086310
3            3        0.076748
4            4        0.069098
5            5        0.072802
6            6        0.072438
7            7        0.090007
8            8        0.101924
9            9        0.123149
Generated_plotly_code:
import plotly.express as px

fig = px.line(df

In [95]:
question = "Show me the hourly trend of rides for each borough on Jan 20th along with running total of rides."
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'pickup_hour', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
SELECT 
    pickup_hour, 
    Borough_PU, 
    total_rides, 
    SUM(total_rides) OVER (PARTITION BY Borough_PU ORDER BY pickup_hour) AS running_total_rides
FROM 
    reporting_data_trips
WHERE 
    DATE(pickup_date) = '2023-01-20'
ORDER BY 
    Borough_PU, pickup_hour;
Displaying few records from the result generated by SQL Query
Empty DataFrame
Columns: [pickup_hour, Borough_PU, total_rides, running_total_rides]
Index: []
Generated_plotly_code:
import plotly.express as px

fig = px.line(df, x='pickup_hour', y='total_rides', color='Borough_PU', 
              title='Hourly Trend of Rides for Each Borough on Jan 20th',
         

In [65]:
question = "Show me the hourly trend of rides for each borough on Jan 20th along with running total of rides. SQL output should look like hour,borough,rides,running sum of rides"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'pickup_hour', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
SELECT 
    pickup_hour AS hour,
    Borough_PU AS borough,
    SUM(total_rides) AS rides,
    SUM(SUM(total_rides)) OVER (PARTITION BY Borough_PU ORDER BY pickup_hour) AS running_sum_of_rides
FROM 
    reporting_data_trips
WHERE 
    DATE(pickup_date) = '2023-01-20'
GROUP BY 
    pickup_hour, Borough_PU
ORDER BY 
    pickup_hour, Borough_PU;
Displaying few records from the result generated by SQL Query
Empty DataFrame
Columns: [hour, borough, rides, running_sum_of_rides]
Index: []


In [66]:
question = "Show me the hourly trend of rides for each borough on Jan 20, 2026 along with running total of rides. SQL output should look like hour,borough,rides,running sum of rides"
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'pickup_hour', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
SELECT 
    pickup_hour AS hour,
    Borough_PU AS borough,
    SUM(total_rides) AS rides,
    SUM(SUM(total_rides)) OVER (PARTITION BY Borough_PU ORDER BY pickup_hour) AS running_sum_of_rides
FROM 
    reporting_data_trips
WHERE 
    DATE(pickup_date) = '2026-01-20'
GROUP BY 
    pickup_hour, Borough_PU
ORDER BY 
    pickup_hour, Borough_PU;
Displaying few records from the result generated by SQL Query
   hour    borough  rides  running_sum_of_rides
0     0      Bronx     13                    13
1     0   Brooklyn     83                    83
2     0  Manhattan   1045                  1045
3     0     Queens    516            

In [67]:
question = "Show me the hourly trend of rides for each borough on Jan 20, 2026 along with running total of rides. SQL output should look like hour,borough,rides,running sum of rides. the visual should show rides as bars and running sum of rides as line in a single plot."
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'pickup_hour', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
SELECT 
    pickup_hour AS hour,
    Borough_PU AS borough,
    SUM(total_rides) AS rides,
    SUM(SUM(total_rides)) OVER (PARTITION BY Borough_PU ORDER BY pickup_hour) AS running_sum_of_rides
FROM 
    reporting_data_trips
WHERE 
    DATE(pickup_date) = '2026-01-20'
GROUP BY 
    pickup_hour, Borough_PU
ORDER BY 
    pickup_hour, Borough_PU;
Displaying few records from the result generated by SQL Query
   hour    borough  rides  running_sum_of_rides
0     0      Bronx     13                    13
1     0   Brooklyn     83                    83
2     0  Manhattan   1045                  1045
3     0     Queens    516            

In [68]:
question = "Show me the hourly trend of rides for each borough on Jan 20, 2026 along with running total of rides. SQL output should look like hour,borough,rides,running sum of rides. the visual should show rides as bars and running sum of rides as line in a separate plot for each borough."
output_df = query_to_sql_to_chart(question ,DATABASE_PATH)

{'parsed_question': {'is_relevant': True, 'relevant_tables': [{'table_name': 'reporting_data_trips', 'columns': ['pickup_date', 'pickup_hour', 'distance_bucket', 'payment_type', 'Borough_PU', 'total_rides', 'total_tip_amount', 'total_fare_amount', 'total_amount'], 'relevant_columns': ['pickup_date', 'pickup_hour', 'Borough_PU', 'total_rides']}]}}
Generated SQL:
SELECT 
    pickup_hour AS hour,
    Borough_PU AS borough,
    SUM(total_rides) AS rides,
    SUM(SUM(total_rides)) OVER (PARTITION BY Borough_PU ORDER BY pickup_hour) AS running_sum_of_rides
FROM 
    reporting_data_trips
WHERE 
    DATE(pickup_date) = '2026-01-20'
GROUP BY 
    pickup_hour, Borough_PU
ORDER BY 
    Borough_PU, pickup_hour;
Displaying few records from the result generated by SQL Query
   hour borough  rides  running_sum_of_rides
0     5    None      2                     2
1     7    None      2                     4
2    17    None      1                     5
3    22    None      3                     8
4   

In [ ]:
# Sample questions to test further

question = "Show me the total rides volume for queens for each month"
question = "Find total rides volume in the last 7 days bases on the maxium date as per the dataset"
question = "Find running sum of total rides."
question = "Find week over week total rides" 
question = "Create a new field tip percentage that is total tip amount as a percentage of total amount" 

Generated SQL:
SELECT strftime('%Y-%m', pickup_date) AS month, SUM(total_rides) AS total_rides_volume
FROM reporting_data_trips
WHERE Borough_PU = 'Queens'
GROUP BY month


In [62]:
import base64
from IPython.display import HTML

# 1. Create your DataFrame
df = output_df

# 2. Function to generate an HTML download link
def create_download_link(dataframe, title="Download CSV file", filename="data.csv"):
    csv = dataframe.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()  # strings <-> bytes conversions
    html = f'<a download="{filename}" href="data:text/csv;base64,{b64}" target="_blank">{title}</a>'
    return HTML(html)

# 3. Display the link
create_download_link(df)